## Chebyshev

In [ ]:
from scipy.signal import cheby1, filtfilt

def chebyshev_bandpass_filter(data, lowcut, highcut, fs, order=3, ripple=0.5):
    """
    Chebyshev Type I band-pass filter (zero-phase)

    Parameters
    ----------
    data : 1D np.ndarray
        Input signal
    lowcut : float
        Low cutoff frequency (Hz)
    highcut : float
        High cutoff frequency (Hz)
    fs : float
        Sampling frequency (Hz)
    order : int
        Filter order (2 is a good default)
    ripple : float
        Passband ripple (dB), typically 0.3–1.0

    Returns
    -------
    y : 1D np.ndarray
        Filtered signal
    """
    b, a = cheby1(
        order,
        ripple,
        [lowcut, highcut],
        btype='band',
        fs=fs
    )
    return filtfilt(b, a, data)

## Cosine Similarity

In [ ]:
# @title
"""
PPG Cosine Similarity Filter (Colab Version)
=============================================
Filters pulse segments based on cosine similarity to the mean template
computed from valid segments (passed H1/H2 and notch filters).

Logic:
1. For each (caseid, glucose_time), get all segments that passed valid_h1_h2 AND valid_notch
2. Compute mean template from these valid segments (padded to 100 samples)
3. Compute cosine similarity of each valid segment against the template
4. Add cosine_similarity column to metadata

Usage:
    - Modify CONFIG section
    - Run the script
"""

import os
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.linalg import norm
from tqdm import tqdm

from scipy.signal import resample

# ==================== CONFIG ====================

# Input paths
METADATA_CSV = '/content/drive/MyDrive/2025_PPG_GLUC/Data/100Hz_ppg_features_4min_BW_spline_metadata.csv'
NPY_DIR = '/content/drive/MyDrive/2025_PPG_GLUC/Data/Processed Data/100Hz_4min_windows/BW_spline/'

# Output paths
OUTPUT_CSV = '/content/drive/MyDrive/2025_PPG_GLUC/Data/100Hz_ppg_features_4min_BW_spline_metadata_cosine_all.csv'
OUTPUT_DIR = '/content/drive/MyDrive/2025_PPG_GLUC/Data/QC_Inspection/cosine_filter/'

# Cosine similarity parameters
TARGET_PULSE_LEN = 100  # Pad/crop pulses to this length
SIMILARITY_THRESHOLD = 0.85  # Per-segment threshold

# Set to True to generate plots for specific cases
GENERATE_PLOTS = False

# Test specific case (set to None to process all)
TEST_CASEID = 2801  # e.g., 4270
TEST_GLUCOSE_TIME = 1633  # e.g., 1950

# Limit number of recordings to process (set to None for all)
PROCESS_LIMIT = None

# Randomize which recordings to process (if PROCESS_LIMIT is set)
RANDOMIZE = False

# ================================================


def cosine_similarity(a, b):
    """
    Compute cosine similarity between two vectors.

    cos_sim = (a · b) / (||a|| × ||b||)

    Returns value in range [-1, 1], where 1 = identical, 0 = orthogonal
    """
    a = np.asarray(a).flatten()
    b = np.asarray(b).flatten()

    norm_a = norm(a)
    norm_b = norm(b)

    if norm_a == 0 or norm_b == 0:
        return 0.0

    return np.dot(a, b) / (norm_a * norm_b)


def pad_or_crop_pulse(pulse, target_len=100, pad_value=0.0):
    """Pad or crop a single pulse to target length."""
    pulse = np.asarray(pulse).flatten()

    if len(pulse) >= target_len:
        return pulse[:target_len]
    else:
        # pad_width = target_len - len(pulse)
        # return np.pad(pulse, (0, pad_width), mode='constant', constant_values=pad_value)
        resampled = resample(pulse, target_len)
        return resampled


def extract_segment(signal, start, end):
    """Extract a segment from signal given start and end indices."""
    signal = np.asarray(signal).flatten()
    start, end = int(start), int(end)

    if end > start and start >= 0 and end <= len(signal):
        return signal[start:end+1]  # +1 to include end index
    return None


def compute_mean_template_from_valid_segments(signal, valid_segments_df, target_len=100):
    """
    Compute mean template from valid segments only.

    Parameters:
    -----------
    signal : np.ndarray
        Full 4-minute signal
    valid_segments_df : pd.DataFrame
        DataFrame containing only valid segments (passed H1/H2 and notch)
        Must have 'valley_start' and 'valley_end' columns
    target_len : int
        Length to pad/crop each segment to

    Returns:
    --------
    np.ndarray : Mean template of shape (target_len,)
    int : Number of segments used to compute template
    """
    signal = np.asarray(signal).flatten()

    segments = []
    for _, row in valid_segments_df.iterrows():
        segment = extract_segment(signal, row['valley_start'], row['valley_end'])
        if segment is not None:
            padded = pad_or_crop_pulse(segment, target_len=target_len)
            segments.append(padded)

    if len(segments) == 0:
        return np.zeros(target_len), 0

    segments = np.array(segments)
    mean_template = np.mean(segments, axis=0)

    return mean_template, len(segments)


def process_single_recording(caseid, glucose_time, df_recording, npy_dir, target_len=100):
    """
    Process a single recording (caseid, glucose_time) and compute cosine similarity
    for each segment against the mean template.

    Returns:
    --------
    pd.DataFrame : Updated dataframe with cosine_similarity column
    dict : Info about the processing (template, stats, etc.)
    """
    # Load signal
    npy_filename = f"case_{caseid}_time_{glucose_time}_cleaned_filtered.npy"
    npy_path = os.path.join(npy_dir, npy_filename)

    if not os.path.exists(npy_path):
        print(f"❌ NPY file not found: {npy_path}")
        return df_recording, None

    signal = np.load(npy_path)
    if len(signal.shape) > 1:
        signal = signal.flatten()

    # Get valid segments (passed both filters)
    valid_mask = (df_recording['valid_h1_h2'] == True) & (df_recording['valid_notch'] == True)
    valid_segments_df = df_recording[valid_mask]

    if len(valid_segments_df) == 0:
        # No valid segments, set all cosine similarities to NaN
        df_recording = df_recording.copy()
        df_recording['cosine_similarity'] = np.nan
        df_recording['valid_cosine'] = False
        return df_recording, {'n_valid': 0, 'template': None}

    # Compute mean template from valid segments only
    mean_template, n_template_segments = compute_mean_template_from_valid_segments(
        signal, valid_segments_df, target_len=target_len
    )

    # Compute cosine similarity for ALL segments (including invalid ones)
    cosine_scores = []
    for _, row in df_recording.iterrows():
        segment = extract_segment(signal, row['valley_start'], row['valley_end'])
        if segment is not None:
            padded = pad_or_crop_pulse(segment, target_len=target_len)
            sim = cosine_similarity(padded, mean_template)
            cosine_scores.append(sim)
        else:
            cosine_scores.append(np.nan)

    # Add columns
    df_recording = df_recording.copy()
    df_recording['cosine_similarity'] = cosine_scores
    df_recording['valid_cosine'] = df_recording['cosine_similarity'] >= SIMILARITY_THRESHOLD

    info = {
        'n_valid_segments': len(valid_segments_df),
        'n_template_segments': n_template_segments,
        'mean_template': mean_template,
        'cosine_mean': np.nanmean(cosine_scores),
        'cosine_min': np.nanmin(cosine_scores),
        'cosine_max': np.nanmax(cosine_scores),
        'n_passed_cosine': sum(df_recording['valid_cosine'])
    }

    return df_recording, info


def plot_15_segments_concatenated(df_recording, caseid, glucose_time, signal, output_dir=None):
    """
    Plot the 15 valid segments concatenated as one continuous waveform.
    Shows both raw concatenation and padded version (what goes into the model).
    """
    # Get valid segments (passed both filters)
    valid_mask = (df_recording['valid_h1_h2'] == True) & (df_recording['valid_notch'] == True)
    valid_df = df_recording[valid_mask].head(15)

    if len(valid_df) < 15:
        print(f"⚠️ Only {len(valid_df)} valid segments available (need 15)")
        return None

    # Extract segments
    raw_segments = []
    padded_segments = []
    segment_indices = []
    original_lengths = []

    for _, row in valid_df.iterrows():
        segment = extract_segment(signal, row['valley_start'], row['valley_end'])
        if segment is not None:
            raw_segments.append(segment)
            padded_segments.append(pad_or_crop_pulse(segment, TARGET_PULSE_LEN))
            segment_indices.append(row['segment_index'])
            original_lengths.append(len(segment))

    # Concatenate
    raw_concat = np.concatenate(raw_segments)
    padded_concat = np.concatenate(padded_segments)

    fig, axes = plt.subplots(3, 1, figsize=(16, 10))

    # 1. Raw concatenated (variable length segments)
    ax1 = axes[0]
    ax1.plot(raw_concat, 'b-', linewidth=0.8)

    # Add vertical lines at segment boundaries
    boundaries = np.cumsum([0] + original_lengths[:-1])
    for i, b in enumerate(boundaries[1:], 1):
        ax1.axvline(x=b, color='r', linestyle='--', alpha=0.5, linewidth=0.5)

    ax1.set_title(f'Raw Concatenated (Variable Length) - Total: {len(raw_concat)} samples', fontsize=11)
    ax1.set_xlabel('Sample')
    ax1.set_ylabel('Amplitude')
    ax1.grid(True, alpha=0.3)

    # 2. Padded concatenated (fixed 100 samples each = 1500 total)
    ax2 = axes[1]
    ax2.plot(padded_concat, 'b-', linewidth=0.8)

    # Add vertical lines at segment boundaries (every 100 samples)
    for i in range(1, 15):
        ax2.axvline(x=i * TARGET_PULSE_LEN, color='r', linestyle='--', alpha=0.5, linewidth=0.5)

    ax2.set_title(f'Padded Concatenated (100 samples each) - Total: {len(padded_concat)} samples', fontsize=11)
    ax2.set_xlabel('Sample')
    ax2.set_ylabel('Amplitude')
    ax2.grid(True, alpha=0.3)

    # 3. Normalized + Padded (what goes into model)
    ax3 = axes[2]

    # Min-max normalize
    normalized = (padded_concat - np.min(padded_concat)) / (np.max(padded_concat) - np.min(padded_concat) + 1e-6)
    ax3.plot(normalized, 'b-', linewidth=0.8)

    # Add vertical lines at segment boundaries
    for i in range(1, 15):
        ax3.axvline(x=i * TARGET_PULSE_LEN, color='r', linestyle='--', alpha=0.5, linewidth=0.5)

    ax3.set_title(f'Normalized + Padded (Model Input) - Total: {len(normalized)} samples', fontsize=11)
    ax3.set_xlabel('Sample')
    ax3.set_ylabel('Amplitude (0-1)')
    ax3.grid(True, alpha=0.3)

    # Add info text
    info_text = f"Segment Indices: {segment_indices}\nOriginal Lengths: {original_lengths}"
    fig.text(0.5, 0.02, info_text, ha='center', fontsize=9, fontfamily='monospace')

    plt.suptitle(f'15-Segment Window - Case {caseid} | Glucose Time {glucose_time}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])

    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        save_path = os.path.join(output_dir, f'case_{caseid}_time_{glucose_time}_15segments_concat.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")

    plt.show()
    return fig


def plot_15_segments_vs_template(df_recording, info, caseid, glucose_time, signal, output_dir=None):
    """
    Plot the 15 valid concatenated segments against the mean template.
    Only plots segments that passed both H1/H2 and notch filters.
    """
    # Get valid segments (passed both filters)
    valid_mask = (df_recording['valid_h1_h2'] == True) & (df_recording['valid_notch'] == True)
    valid_df = df_recording[valid_mask].head(15)  # Take first 15 valid segments

    if len(valid_df) < 15:
        print(f"⚠️ Only {len(valid_df)} valid segments available (need 15)")
        return None

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # Extract and pad the 15 segments
    padded_segments = []
    cosine_scores = []
    segment_indices = []

    for _, row in valid_df.iterrows():
        segment = extract_segment(signal, row['valley_start'], row['valley_end'])
        if segment is not None:
            padded = pad_or_crop_pulse(segment, TARGET_PULSE_LEN)
            padded_segments.append(padded)
            cosine_scores.append(row['cosine_similarity'])
            segment_indices.append(row['segment_index'])

    # 1. Mean template
    ax1 = axes[0, 0]
    ax1.plot(info['mean_template'], 'b-', linewidth=2)
    ax1.set_title(f'Mean Template (from {info["n_template_segments"]} valid segments)', fontsize=11)
    ax1.set_xlabel('Sample')
    ax1.set_ylabel('Amplitude')
    ax1.grid(True, alpha=0.3)

    # 2. All 15 segments overlaid with template
    ax2 = axes[0, 1]
    for i, (pulse, cos_sim) in enumerate(zip(padded_segments, cosine_scores)):
        color = 'blue' if cos_sim >= SIMILARITY_THRESHOLD else 'red'
        alpha = 0.4 if cos_sim >= SIMILARITY_THRESHOLD else 0.8
        ax2.plot(pulse, color=color, alpha=alpha, linewidth=0.8, label=f'Seg {segment_indices[i]}' if i < 5 else None)
    ax2.plot(info['mean_template'], 'k-', linewidth=2.5, label='Mean Template')
    ax2.set_title('15 Valid Segments vs Template (Red = below 0.85)', fontsize=11)
    ax2.set_xlabel('Sample')
    ax2.set_ylabel('Amplitude')
    ax2.legend(loc='upper right', fontsize=8)
    ax2.grid(True, alpha=0.3)

    # 3. Individual similarity scores bar chart
    ax3 = axes[1, 0]
    colors = ['green' if s >= SIMILARITY_THRESHOLD else 'red' for s in cosine_scores]
    bars = ax3.bar(range(len(cosine_scores)), cosine_scores, color=colors, alpha=0.8)
    ax3.axhline(y=SIMILARITY_THRESHOLD, color='orange', linestyle='--', linewidth=2,
                label=f'Threshold ({SIMILARITY_THRESHOLD})')
    ax3.set_xlabel('Pulse Index (0-14)')
    ax3.set_ylabel('Cosine Similarity')
    ax3.set_title('Cosine Similarity of 15 Valid Segments', fontsize=11)
    ax3.set_xticks(range(15))
    ax3.set_xticklabels([f'{i}\n({seg_indices})' for i, seg_indices in enumerate(segment_indices)], fontsize=7)
    ax3.legend()
    ax3.grid(True, alpha=0.3, axis='y')

    # 4. Summary
    ax4 = axes[1, 1]
    ax4.axis('off')

    total_score = sum(cosine_scores)
    threshold_total = SIMILARITY_THRESHOLD * 15
    n_passed = sum(1 for s in cosine_scores if s >= SIMILARITY_THRESHOLD)
    n_failed = 15 - n_passed
    status = "✅ PASS" if total_score >= threshold_total else "❌ FAIL"

    summary_text = f"""
    Case ID: {caseid}
    Glucose Time: {glucose_time}

    15 Valid Segments (passed H1/H2 & Notch):
    Segment Indices: {segment_indices}

    Cosine Similarity Scores:
    {np.round(cosine_scores, 4)}

    Individual Stats:
      Min:  {min(cosine_scores):.4f}
      Max:  {max(cosine_scores):.4f}
      Mean: {np.mean(cosine_scores):.4f}

    Total Score: {total_score:.4f}
    Threshold:   {threshold_total:.2f} (0.85 × 15)

    Status: {status}

    Passed (>= {SIMILARITY_THRESHOLD}): {n_passed} / 15
    Failed (< {SIMILARITY_THRESHOLD}): {n_failed} / 15
    """

    ax4.text(0.05, 0.95, summary_text, transform=ax4.transAxes, fontsize=10,
             verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.suptitle(f'15 Valid Segments Analysis - Case {caseid} | Glucose Time {glucose_time}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()

    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        save_path = os.path.join(output_dir, f'case_{caseid}_time_{glucose_time}_15segments_vs_template.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")

    plt.show()
    return fig


def plot_cosine_analysis(df_recording, info, caseid, glucose_time, signal, output_dir=None):
    """
    Plot detailed cosine similarity analysis for a recording.
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # Get valid segments for plotting
    valid_mask = (df_recording['valid_h1_h2'] == True) & (df_recording['valid_notch'] == True)
    valid_df = df_recording[valid_mask]

    # 1. Mean template
    ax1 = axes[0, 0]
    ax1.plot(info['mean_template'], 'b-', linewidth=1.5)
    ax1.set_title(f'Mean Template (from {info["n_template_segments"]} valid segments)', fontsize=11)
    ax1.set_xlabel('Sample')
    ax1.set_ylabel('Amplitude')
    ax1.grid(True, alpha=0.3)

    # 2. Valid segments overlaid with template
    ax2 = axes[0, 1]
    for _, row in valid_df.iterrows():
        segment = extract_segment(signal, row['valley_start'], row['valley_end'])
        if segment is not None:
            padded = pad_or_crop_pulse(segment, TARGET_PULSE_LEN)
            cos_sim = row['cosine_similarity']
            color = 'blue' if cos_sim >= SIMILARITY_THRESHOLD else 'red'
            alpha = 0.3 if cos_sim >= SIMILARITY_THRESHOLD else 0.7
            ax2.plot(padded, color=color, alpha=alpha, linewidth=0.5)
    ax2.plot(info['mean_template'], 'k-', linewidth=2, label='Template')
    ax2.set_title('Valid Segments vs Template (Red = below threshold)', fontsize=11)
    ax2.set_xlabel('Sample')
    ax2.set_ylabel('Amplitude')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    # 3. Cosine similarity distribution (valid segments only)
    ax3 = axes[1, 0]
    valid_cosines = valid_df['cosine_similarity'].dropna()
    colors = ['green' if s >= SIMILARITY_THRESHOLD else 'red' for s in valid_cosines]
    ax3.bar(range(len(valid_cosines)), valid_cosines, color=colors, alpha=0.7)
    ax3.axhline(y=SIMILARITY_THRESHOLD, color='orange', linestyle='--', linewidth=2,
                label=f'Threshold ({SIMILARITY_THRESHOLD})')
    ax3.set_xlabel('Valid Segment Index')
    ax3.set_ylabel('Cosine Similarity')
    ax3.set_title(f'Cosine Similarity of Valid Segments (n={len(valid_cosines)})', fontsize=11)
    ax3.legend()
    ax3.grid(True, alpha=0.3, axis='y')

    # 4. Summary
    ax4 = axes[1, 1]
    ax4.axis('off')

    n_valid = len(valid_df)
    n_passed_cosine = valid_df['valid_cosine'].sum()
    n_failed_cosine = n_valid - n_passed_cosine

    summary_text = f"""
    Case ID: {caseid}
    Glucose Time: {glucose_time}

    Total Segments: {len(df_recording)}
    Valid (H1/H2 & Notch): {n_valid}

    Cosine Similarity Stats (valid segments):
      Mean: {info['cosine_mean']:.4f}
      Min:  {info['cosine_min']:.4f}
      Max:  {info['cosine_max']:.4f}

    Threshold: {SIMILARITY_THRESHOLD}
    Passed Cosine: {n_passed_cosine} ({100*n_passed_cosine/n_valid:.1f}%)
    Failed Cosine: {n_failed_cosine} ({100*n_failed_cosine/n_valid:.1f}%)

    Final Valid (H1/H2 & Notch & Cosine): {n_passed_cosine}
    """

    ax4.text(0.1, 0.9, summary_text, transform=ax4.transAxes, fontsize=11,
             verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.suptitle(f'Cosine Similarity Analysis - Case {caseid} | Glucose Time {glucose_time}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()

    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        save_path = os.path.join(output_dir, f'case_{caseid}_time_{glucose_time}_cosine_analysis.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")

    plt.show()
    return fig


def process_all_recordings(metadata_csv, npy_dir, output_csv, output_dir=None, generate_plots=False, limit=None, randomize=False):
    """
    Process all recordings and add cosine similarity to metadata.

    Parameters:
    -----------
    limit : int, optional
        Limit number of recordings to process (for testing)
    randomize : bool
        If True, randomly select recordings instead of first N
    """
    print(f"Loading metadata: {metadata_csv}")
    df = pd.read_csv(metadata_csv)
    print(f"Total segments: {len(df)}")

    # Group by (caseid, glucose_time)
    groups = list(df.groupby(['caseid', 'glucose_time']))
    print(f"Total recordings: {len(groups)}")

    # Apply limit if specified
    if limit is not None:
        if randomize:
            import random
            random.seed()  # Use random seed for different results each run
            groups = random.sample(groups, min(limit, len(groups)))
            print(f"Randomly selected {len(groups)} recordings")
        else:
            groups = groups[:limit]
            print(f"Processing first {limit} recordings only")

    all_results = []

    for (caseid, glucose_time), group_df in tqdm(groups, desc="Processing recordings"):
        try:
            result_df, info = process_single_recording(
                caseid=int(caseid),
                glucose_time=int(glucose_time),
                df_recording=group_df,
                npy_dir=npy_dir,
                target_len=TARGET_PULSE_LEN
            )

            all_results.append(result_df)

            # Generate plot if requested
            if generate_plots and info is not None and info.get('mean_template') is not None:
                npy_path = os.path.join(npy_dir, f"case_{int(caseid)}_time_{int(glucose_time)}_cleaned_filtered.npy")
                if os.path.exists(npy_path):
                    signal = np.load(npy_path).flatten()

                    # Plot concatenated 15-segment window
                    plot_15_segments_concatenated(result_df, caseid, glucose_time, signal, output_dir)
                    plt.close()

                    # Plot 15 valid segments vs template
                    plot_15_segments_vs_template(result_df, info, caseid, glucose_time, signal, output_dir)
                    plt.close()

                    # Plot full cosine analysis
                    plot_cosine_analysis(result_df, info, caseid, glucose_time, signal, output_dir)
                    plt.close()

        except Exception as e:
            print(f"Error processing case {caseid}, time {glucose_time}: {e}")
            group_df = group_df.copy()
            group_df['cosine_similarity'] = np.nan
            group_df['valid_cosine'] = False
            all_results.append(group_df)

    # Combine all results
    final_df = pd.concat(all_results, ignore_index=True)

    # Summary statistics
    valid_h1h2_notch = (final_df['valid_h1_h2'] == True) & (final_df['valid_notch'] == True)
    valid_all = valid_h1h2_notch & (final_df['valid_cosine'] == True)

    print(f"\n{'='*60}")
    print("FILTERING SUMMARY")
    print(f"{'='*60}")
    print(f"Total segments: {len(final_df)}")
    print(f"Valid (H1/H2 & Notch): {valid_h1h2_notch.sum()}")
    print(f"Valid (H1/H2 & Notch & Cosine >= {SIMILARITY_THRESHOLD}): {valid_all.sum()}")
    print(f"Removed by cosine filter: {valid_h1h2_notch.sum() - valid_all.sum()}")
    print(f"{'='*60}")

    # Save output
    if output_csv:
        os.makedirs(os.path.dirname(output_csv), exist_ok=True)
        final_df.to_csv(output_csv, index=False)
        print(f"\nSaved results to: {output_csv}")

    return final_df


# ==================== RUN ====================

if __name__ == "__main__":

    if TEST_CASEID is not None and TEST_GLUCOSE_TIME is not None:
        # Test single recording
        print(f"Testing single recording: case {TEST_CASEID}, glucose_time {TEST_GLUCOSE_TIME}")

        df = pd.read_csv(METADATA_CSV)

        # Filter to specific recording
        mask = (df['caseid'] == TEST_CASEID) & (df['glucose_time'] == TEST_GLUCOSE_TIME)
        recording_df = df[mask]

        if len(recording_df) == 0:
            print(f"No matching recording found")
        else:
            print(f"Found {len(recording_df)} segments for this recording")

            result_df, info = process_single_recording(
                caseid=TEST_CASEID,
                glucose_time=TEST_GLUCOSE_TIME,
                df_recording=recording_df,
                npy_dir=NPY_DIR,
                target_len=TARGET_PULSE_LEN
            )

            if info:
                print(f"\n{'='*60}")
                print(f"Template computed from {info['n_template_segments']} valid segments")
                print(f"Cosine similarity stats:")
                print(f"  Mean: {info['cosine_mean']:.4f}")
                print(f"  Min:  {info['cosine_min']:.4f}")
                print(f"  Max:  {info['cosine_max']:.4f}")
                print(f"Passed cosine threshold: {info['n_passed_cosine']}")
                print(f"{'='*60}")

                # Load signal for plotting
                npy_path = os.path.join(NPY_DIR, f"case_{TEST_CASEID}_time_{TEST_GLUCOSE_TIME}_cleaned_filtered.npy")
                if os.path.exists(npy_path):
                    signal = np.load(npy_path).flatten()

                    # Plot concatenated 15-segment window
                    plot_15_segments_concatenated(result_df, TEST_CASEID, TEST_GLUCOSE_TIME, signal, OUTPUT_DIR)

                    # Plot 15 valid segments vs template
                    plot_15_segments_vs_template(result_df, info, TEST_CASEID, TEST_GLUCOSE_TIME, signal, OUTPUT_DIR)

                    # Plot full cosine analysis
                    plot_cosine_analysis(result_df, info, TEST_CASEID, TEST_GLUCOSE_TIME, signal, OUTPUT_DIR)

                # Show results
                valid_mask = (result_df['valid_h1_h2'] == True) & (result_df['valid_notch'] == True)
                print("\nValid segments with cosine scores:")
                print(result_df[valid_mask][['segment_index', 'cosine_similarity', 'valid_cosine']].to_string())

    else:
        # Process all recordings
        final_df = process_all_recordings(
            metadata_csv=METADATA_CSV,
            npy_dir=NPY_DIR,
            output_csv=OUTPUT_CSV,
            output_dir=OUTPUT_DIR if GENERATE_PLOTS else None,
            generate_plots=GENERATE_PLOTS,
            limit=PROCESS_LIMIT,
            randomize=RANDOMIZE
        )

## Sample Entropy

In [ ]:
def sample_entropy(signal, m=2, r=None):
    signal = np.asarray(signal)
    N = len(signal)

    if r is None:
        r = 0.2 * np.std(signal)

    def _phi(m):
        x = np.array([signal[i:i+m] for i in range(N-m+1)])
        C = np.sum(np.max(np.abs(x[:, None] - x[None, :]), axis=2) <= r, axis=0) - 1
        return np.sum(C)

    return -np.log(_phi(m+1) / _phi(m) + 1e-12)

In [ ]:
import os
import numpy as np
import pandas as pd
import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

# Change paths here
base_dir = '/content/drive/MyDrive/2025_PPG_GLUC/Data/Processed Data/100Hz_4min_windows/BW_spline/'
PAD_BEFORE_CONCAT_PATH = '/content/drive/MyDrive/2025_PPG_GLUC/Data/'

THRESH = 0.0
MAX_WORKERS = max(1, (os.cpu_count() or 2) - 1)

def _worker(row_dict, base_dir):
    try:
        caseid = int(row_dict["caseid"])
        glucose_time = int(float(row_dict["glucose_time"]))
        glucose_value = int(row_dict["glucose"])
        segment, length = concat_pulses_from_bin(row_dict, base_dir)
        if segment is None:
            return None

        ent = sample_entropy(segment)

        if ent > THRESH:
            return (caseid, glucose_time, glucose_value, float(ent))
        return None

    except Exception as e:
        return None

def parallel_scan(bin_data: pd.DataFrame, base_dir: str, max_workers: int = MAX_WORKERS):
    exceed_entropy = []

    # Use itertuples -> dict (faster + picklable)
    rows = [r._asdict() for r in bin_data.itertuples(index=False, name="Row")]

    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(_worker, row_dict, base_dir) for row_dict in rows]

        for fut in tqdm.tqdm(as_completed(futures), total=len(futures)):
            res = fut.result()
            if res is None:
                continue

            caseid, glucose_time, glucose_value, ent = res
            if ent > 0.4:
              print(f"Caseid: {caseid}, glucose_time: {glucose_time}"
                    f"glucose_value: {glucose_value}, sampen: {ent:.4f}")
            exceed_entropy.append((caseid, glucose_time, glucose_value, ent))

    return exceed_entropy


# usage
if __name__ == "__main__":
    # Change path to your binned metadata path
    bin_data = pd.read_csv(
        PAD_BEFORE_CONCAT_PATH + "100Hz_ppg_features_4min_BW_Spline_binned_metadata.csv"
    )

    exceed_entropy = parallel_scan(bin_data, base_dir)
    print("Num exceed:", len(exceed_entropy))

## Parkes Error Grid

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def parkes_error_grid(ref_values, pred_values, title="Parkes Consensus Error Grid (Type 1)"):
    ref_values = np.array(ref_values).flatten()
    pred_values = np.array(pred_values).flatten()

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(ref_values, pred_values, marker='o', color='black', s=10, alpha=0.5, label='Predictions')
    ax.set_title(title, fontsize=16)
    ax.set_xlabel("Reference Glucose (mg/dL)", fontsize=14)
    ax.set_ylabel("Predicted Glucose (mg/dL)", fontsize=14)
    ax.set_xlim(0, 550)
    ax.set_ylim(0, 550)
    ax.plot([0, 550], [0, 550], linestyle='--', color='grey', alpha=0.5)

    # Zones
    # Boundary A | B (Upper)
    ab_upper = [(0, 50), (30, 50), (140, 170), (280, 380), (430, 550)]
    bc_upper = [(0, 60), (30, 80), (50, 110), (70, 180), (260, 550)]
    cd_upper = [(0, 100), (25, 100), (50, 125), (80, 215), (125, 550)]
    de_upper = [(0, 150), (35, 155), (50, 550)]
    # Boundary A | B (Lower)
    ab_lower = [(50, 0), (170, 145), (385, 300), (550, 450)]
    bc_lower = [(120, 0), (260, 130), (550, 250)]
    cd_lower = [(250, 0), (250, 40), (550, 150)]

    def plot_line(coords, color='k', linewidth=1.5):
        x, y = zip(*coords)
        ax.plot(x, y, color=color, linewidth=linewidth)

    for line in [ab_upper, ab_lower]: plot_line(line, 'green')
    for line in [bc_upper, bc_lower]: plot_line(line, 'orange')
    for line in [cd_upper, cd_lower]: plot_line(line, 'red')
    plot_line(de_upper, 'purple')

    # Add Zone Labels
    ax.text(275, 275, 'A', fontsize=20, ha='center', va='center', color='green')
    ax.text(275, 450, 'B', fontsize=15, ha='center', color='orange')
    ax.text(120, 450, 'C', fontsize=15, ha='center', color='red')
    ax.text(60, 450, 'D', fontsize=15, ha='center', color='purple')
    ax.text(20, 450, 'E', fontsize=15, ha='center', color='black')

    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    return fig

## Eval Metrics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

def plot_performance(model, loaders, history, best_val_mae, MODEL_SAVE_PATH, device):
    # Now the function 'knows' what model, loaders, and device are
    model.eval()
    preds, trues = [], []

    with torch.no_grad():
        for x, y, lens in loaders['test']:
            x, lens = x.to(device), lens.to(device)
            out = model(x, lens)
            preds.extend(out.cpu().numpy())
            trues.extend(y.numpy())

    preds = np.array(preds).flatten()
    trues = np.array(trues).flatten()

    preds = np.array(preds); trues = np.array(trues)
    # 1. METRICS
    mae = mean_absolute_error(trues, preds)
    rmse = np.sqrt(mean_squared_error(trues, preds))
    mard = np.mean(np.abs((trues - preds) / trues)) * 100
    r2 = r2_score(trues, preds)

    print(f"\n📊 TEST RESULTS:")
    print(f"   RMSE: {rmse:.2f} mg/dL")
    print(f"   MAE:  {mae:.2f} mg/dL")
    print(f"   MARD: {mard:.2f} %")
    print(f"   R^2:  {r2:.4f}")

    # 2. Plotting
    plt.figure(figsize=(14, 6))

    # MAE Subplot
    plt.subplot(1, 2, 1)
    plt.plot(history['t_mae'], label='Train MAE', linewidth=2)
    plt.plot(history['v_mae'], label='Val MAE', linewidth=2)
    plt.plot(history['test_mae'], label='Test MAE', linestyle='--', alpha=0.7)
    plt.title(f'MAE Learning Curve\n(Best Val MAE: {best_val_mae:.2f})')
    plt.xlabel('Epoch'); plt.ylabel('MAE (mg/dL)')
    plt.legend(); plt.grid(True, alpha=0.3)

    # Loss Subplot
    plt.subplot(1, 2, 2)
    plt.plot(history['t_loss'], label='Train Loss')
    plt.plot(history['v_loss'], label='Val Loss')
    plt.plot(history['test_loss'], label='Test Loss', linestyle='--')
    plt.title('Loss (MSE) Learning Curve')
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.legend(); plt.grid(True, alpha=0.3)

    plt.tight_layout()
    os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
    plt.savefig(os.path.join(MODEL_SAVE_PATH, 'learning_curves.png'))
    plt.show()
    return trues, preds